# PRM FTE & Fleet Optimisation Model: S25 Baseline to S26 Forecast
### Scenarios:
1. **Current:** 
2. **Relay 1:1:** Ambulift fo Minibus (single flight)
3. **Relay Optimised:** Ambulift to Minibus (Multi-Stand Pooling)

**Objective:** Minimise Cost (Capex and Opex) while maintaining <1% ECAC SLA Failure 

In [6]:
import sys
from pathlib import Path


# Go up until repo root
repo_root = Path.cwd().parent
sys.path.append(str(repo_root))


import simpy
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import tree
from scipy import stats

from modules.utils.query import query
from modules.utils.dates import add_date_parts, to_datetime
from modules.config import JETBRIDGE_STANDS
from modules.domain.prm.minibus import passenger_level_flags

#Formatting for plots
sns.set_theme(style="whitegrid")

In [7]:
#configurable variables
NO_JETBRIDGE_AIRLINES = ["2S", "3V", "6I", "8H", "AP", "BE", "BGH", "BT", "BY", "C3", "D0", "D8", "DS", "DY", "E4", "E9", "EA", "EC", "ED", "EJU", "EVE", "EW", "FH", "FHY", "FI", "FR", "FX", "GR", "HAT", "HV", "LM", "NPT", "OS", "QS", "RC", "RK", "SK", "SRR", "T3", "TO", "TP", "U2", "V3", "W6", "WF", "WK", "XC", "XQ", "ZT", "LO" ]

#Probability Constants
WCHS_OWN_CHAIR_PROB = 0.109

#SSR numeric mapping
def map_ssr(ssr_code):
    if ssr_code == "WCHC": return 3
    if ssr_code == "WCHS": return 2
    return 1 #all others (WCHR, DNPA etc)
    

#historical comparison dates
#Summer 2025
start_S25 = "2025-03-30"
end_S25   = "2025-10-26"

## Phase 1: Data Ingestion and Assumptions
In this section, we load the S25 Actuals and define the Stochastic variables (the probability distributions) derived from historical data

In [8]:
#Load Data Functions

def load_prm_data(start: str, end: str) -> pd.DataFrame:
    """ 
    Load PRM Data where Billing PRM = 1, for time period 
    
    Parameters
    ----------
    start: str
        Start of window (ISO format)
    end: str
        End of window (ISO format)
    
     Returns
    ----------
    pd.DataFrame
        Dataframe of PRM Data with columns:
        ['Job ID', 'Passenger ID', 'Operation Date', 'Vehicle Type', 'Operation Date_day', 'Operation Date_month', 'Operation Date_year']
    """

    
    start_op = start.replace("-", "")   # "2025-01-01" → "20250101"
    end_op   = end.replace("-", "")     # "2026-01-01" → "20260101"


    df = query(
        table="PRM.CompletedServicesByJob",
        columns = [
            "RequestID AS [Job ID]",
            "PassengerID AS [Passenger ID]",
            "FlightID AS [Flight ID]",
            "AirlineCode_IATA AS [Airline Code]",
            "FlightNumber AS [Flight Number]",
            "Sector",
            "ArrDep AS [A/D]",
            "adhocOrPlanned AS [Adhoc or Planned]",
            "requestCreated_DAteTime_Local AS [Request Created DT]",
            "currentSSRCode AS [SSR Code]",
            "startService_DateTime_Local AS [Job Start Time]",
            "finishService_DateTime_Local AS [Job End Time]",
            "disregardCode AS [Disregard Code]",
            "scheduledPickupLocation AS [Scheduled PU Location]",
            "scheduledDestinationLocation AS [Scheduled DO Location]",
            "actualPickupLocation AS [Actual PU Location]",
            "actualDestinationLocation AS [Actual DO Location]",
            "scheduledPickup_DateTime_Local AS [Scheduled PU DT]",
            "arriveAtLocation_DateTime_Local AS [Location Arrival DT]",
            "arrival_ActualGate_DateTime_Local AS [Plane Gate Arrival DT]",
            "ScheduledDateTime_Local AS [Scheduled Flight DT]",
            "ActualDateTime_Local AS [Actual Flight DT]",
            "EmployeeName AS [Employee]",
            "VehicleShortName AS [Vehicle Model]",
            "VehicleTypeName AS [Vehicle Type]",
            "StandCode AS [Stand]"
        ],
        where = ["BillingPRM = 1",
                 "Operation_DateID_Local >= :start_op",
                 "Operation_DateID_Local < :end_op",
        ],
        params= {"start_op": start_op, "end_op": end_op},
        query_option = "OPTION (RECOMPILE)",
    )
    
    df = to_datetime(df, ["Request Created DT", "Job Start Time", "Job End Time", "Scheduled PU DT", "Location Arrival DT", "Plane Gate Arrival DT", "Scheduled Flight DT", "Actual Flight DT"])
    df = add_date_parts(df, col="Actual Flight DT", day=True)

    df["Flight Number"] = df["Flight Number"].astype(str).str.lstrip("0")

    return df

def load_flight_data(start: str, end: str) -> pd.DataFrame:
    """
    Load flight data from EAL.FlightPerformance for merge.
    """

    df = query(
        table="EAL.FlightPerformance",
        columns=[
            "FlightID AS [Flight ID]",
            "ScheduledDateTime_Local AS [Scheduled DateTime]",
            "ArrDeptureCode AS [A/D]",
            "FlightNumber AS [Flight Number]",
            "AirlineCode_IATA AS [Airline Code]",
            "Sector",
            "StandCode AS [Stand]",
            "DepartureGate AS [Departure Gate]",
            "ActualDateTime_Local AS [Actual Flight DT]",
            "ChocksDateTime_Local AS [Chocks DT]",
            "BoardingStartDateTime_Local AS [Boarding Start Time]",
            "BoardingEndDateTime_Local AS [Boarding End Time]",
            "TurnAroundFlightNumber AS [Turnaround Flight Number]",
            "TurnAroundStandCode AS [Turnaround Stand Code]",
            "TurnAround_ScheduledDateTime_Local AS [Turnaround Scheduled DT]",
            "TurnAround_Actual_Local AS [Turnaround Actual DT]",
            "Turnaround_Chocks_LOcal AS [Turnaround Chocks]",
            "MinutesOnStand_Chocks AS [Minutes on Chocks]",
            "RemoteStand AS [Remote Stand]",
            "IsPassengerFlight"
        ],
        where=[
            "ScheduledDateTime_Local >= :start",
            "ScheduledDateTime_Local < :end",
        ],
        params={"start": start, "end": end},
        query_option="OPTION (RECOMPILE)",
    )

    df = to_datetime(df, ["Scheduled DateTime", "Actual Flight DT", "Chocks DT", "Boarding Start Time", "Boarding End Time", "Turnaround Scheduled DT", "Turnaround Actual DT", "Turnaround Chocks"])
    df = add_date_parts(df, col="Actual Flight DT", day=True)

    df["Flight Number"] = df["Flight Number"].astype(str).str.lstrip("0")

    return df


In [9]:
#Load Flight Data
df_flights = load_flight_data(start_S25, end_S25)

df_flights = df_flights.sort_values("Chocks DT").reset_index(drop=True)

#2. Effective Remote Flag
df_flights["IsEffectiveRemote"] = np.where(
    (df_flights["Remote Stand"] ==1) |
    (df_flights["Airline Code"].isin(NO_JETBRIDGE_AIRLINES)),
    1, 0
)

def calculate_rolling_stress(row , df):
    #Create a +- 30 minute window
    start_window = row["Chocks DT"] - timedelta(minutes=30)
    end_window = row["Chocks DT"] + timedelta(minutes=30)

    #Count all flights that fall inside specific box
    concurrent_count = len(df[(df["Chocks DT"] >= start_window) &
                            (df["Chocks DT"] <= end_window)])
    
    #subtract 1 so flight itself isnt counted
    return concurrent_count - 1

df_flights['Concurrent Stress'] = df_flights.apply(lambda x: calculate_rolling_stress(x, df_flights), axis=1)



In [ ]:
#Load PRM Data
df_prm = load_prm_data(start_S25, end_S25)

df_prm["SSR numeric"] = df_prm["SSR Code"].apply(map_ssr)

#Assign passenger flags to prm_df
df_prm_flags = passenger_level_flags(df_prm)

np.random.seed(42)
rolls = np.random.rand(len(df_prm_flags))

#Add has own chair column
df_prm_flags["Has Own Chair"] = 0
df_prm_flags.loc[df_prm_flags["SSR numeric"] ==3, "Has Own Chair"] = 1
df_prm_flags.loc[(df_prm_flags["SSR numeric"] ==2) & (rolls < WCHS_OWN_CHAIR_PROB), "Has Own Chair"] = 1

#Condition column
#If arrival we want to know where they STARTED
#If departure we want to know where they ended

df_prm_flags['Strategic Location'] = np.where(
    df_prm_flags["Sector"] == "A",
    df_prm_flags["Actual PU Location"],
    df_prm_flags["Actual DO Location"]
)

agg_rules = {
    "Sector": 'first',
    "A/D": 'first',
    "Adhoc Or Planned": 'first',
    "SSR Code": 'first',
    "SSR numeric": 'first',
    "Has Own Chair": 'max',
    "Job Start Time": 'min',
    "Job End Time": 'max',
    "Strategic Location": 'first',
    "Location Arrival DT": 'min',
    "Plane Gate Arrival DT": 'first',
    "Scheduled Flight DT": 'first',
    "Stand": 'first',
    "PassengerType" : 'first'
}

df_prm_grouped = df_prm.groupby(["Passenger ID", "Airline Code", "Flight Number"]).agg(agg_rules).reset_index()

#Calculate PRMs per flight
flight_prm_count = df_prm_flags.groupby(["Flight Number", "Airline Code", "Day"])['Passenger ID'].nunique().reset_index(name="PRM Flight Count")

df_flights = pd.merge(df_flights, flight_prm_count, on=["Flight Number", "Airline Code", "Day"], how="left").fillna({'PRM Flight Count':0})

#merge with flight data
df_prm_master = pd.merge(
    df_prm_flags, 
    df_flights[
        ["Flight Number", "Airline Code", "Day", "IsEffectiveRemote", "Concurrent Stress", "Minutes on Chocks"]
    ],
    on = ["Flight Number", "Airline Code", "Day"],
    how = "left"
)

#check for PRMs that didn't find flight
orphans = df_prm_master['IsEffectiveRemote'].isna().sum()
print(f"Waring: {orphans} passengers could not be matched to a flight")

print(df_prm_master["PRM Flight Count"].value_counts())

### Operational Asset Allocation Analysis: S25 Baseline

This analysis extracts historical dispatch logic from Summer 2025 (S25) data to understand the allocation of Ambulifts and Minibuses

Objective:
- Identify Dispatch Rules
- Baseline for Scenarios
- FTE Impact